In [ ]:
import pandas as pd 
import seaborn as sns
from  sklearn.preprocessing import StandardScaler

In [ ]:
data = pd.read_csv("smartcart_customers.csv")
data.head()
data.info()
data.isnull().sum()

In [ ]:
income_df = pd.DataFrame(data["Income"])
income_df 

In [ ]:
## Handle missing values
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(
    strategy="median"
)
data["Income"] = imputer.fit_transform(data[["Income"]])
data.isnull().sum()

In [ ]:
data.head()

# Feature Engineering

In [ ]:
data.columns

In [ ]:
## make a new column Age
data["Age"] =2026 - data["Year_Birth"]


In [ ]:
## Handle Data of the customes join the platform 
data["Dt_Customer"].dtype
data["Dt_Customer"] = pd.to_datetime(data["Dt_Customer"] , dayfirst=True) # Pandas expect month-day-year
ref_date = data["Dt_Customer"].max()
data["Customer_Tenure_days"] = (ref_date - data["Dt_Customer"]).dt.days

In [ ]:
data.head()

In [ ]:
data.columns

In [ ]:
## make all spendings of the customers in one column
data["Total_spending"] = data["MntWines"]+ data["MntFruits"] + data["MntMeatProducts"]+data["MntSweetProducts"] + data["MntFishProducts"] + data["MntGoldProds"]

In [ ]:
data.head()

In [ ]:
## make one column for the children
data["Total_children"] = data["Kidhome"]+data["Kidhome"]

In [ ]:
data["Education"].value_counts()

In [ ]:
data["Education"] = data["Education"].replace({
    "2n Cycle": "Undergraduate",
    "Basic": "Undergraduate",
    "Graduation": "Graduate",
    "Master": "PostGraduate",
    "PhD": "PostGraduate"
})

In [ ]:
data.columns

In [ ]:
data["Marital_Status"].value_counts()

In [ ]:
## Make marital status in diff cstegories
data["Marital_Status"] = data["Marital_Status"].replace({
    "Married": "Partner",
    "Together": "Partner",
    "Single": "Single",
    "Divorced": "Single",
    "Widow": "Single",
    "Alone": "Single",
    "Absurd": "Single",
    "YOLO": "Single"
})

In [ ]:
data.head()

## Remove Unnecessary Feature

In [ ]:
data.head()

In [ ]:
cols = ["ID","Year_Birth","Kidhome","Teenhome","Dt_Customer"]
spending_columns = ["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]
total_drop_cols = cols+spending_columns
cleaned_df = data.drop(columns=total_drop_cols)
cleaned_df.shape

In [ ]:
## ckecking for outliers
cols=["Income","Recency","Age","Total_spending","Total_children","Complain","Customer_Tenure_days"]
sns.pairplot(data[cols])

In [ ]:
## remove the outlers uding the IQR meethd
num_cols = ["Income", "Age", "Total_spending"]

for col in num_cols:
    Q1 = cleaned_df[col].quantile(0.25)
    Q3 = cleaned_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    cleaned_df = cleaned_df[(cleaned_df[col] >= lower) & (cleaned_df[col] <= upper)]

print(cleaned_df.shape)

In [ ]:
## Rechecking for the outliers
cols=["Income","Recency","Age","Total_spending","Total_children","Complain","Customer_Tenure_days"]
sns.pairplot(cleaned_df[cols])

In [ ]:
## Plotting correlation heatmap
corr = cleaned_df.corr(numeric_only=True)
corr

In [ ]:

sns.heatmap(
    corr,
    cmap="coolwarm",
    annot_kws ={"size":6},
    annot=True
)

In [ ]:
## Encode our categorical Feature
cleaned_df.head()
cleaned_df.shape

# Data Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
cleaned_df.columns

In [ ]:
ohe = OneHotEncoder()
cat_cols = ["Education","Marital_Status"]
enc_cols = ohe.fit_transform(cleaned_df[cat_cols])

In [ ]:
enc_cols_df = pd.DataFrame(enc_cols.toarray() , columns=ohe.get_feature_names_out(cat_cols),index=cleaned_df.index)

In [ ]:
enc_cols_df.head()

In [ ]:
encoded_df = pd.concat([cleaned_df.drop(columns=cat_cols),enc_cols_df],axis=1)

In [ ]:
encoded_df.shape
encoded_df.head()

## Scalling

In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(encoded_df)

In [ ]:
df_scaled

## Choosing the value of the k (Elbow & Sillhoute Score)

In [ ]:
## Doing pca
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Apply PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(df_scaled)


# Create DataFrame (optional)
import pandas as pd
pca_df = pd.DataFrame(
    pca_result,
    columns=['PC1', 'PC2', 'PC3']
)
# Explained variance
print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)
print("Total Variance Explained:",
      pca.explained_variance_ratio_.sum())

# 3D Plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    pca_df['PC1'],
    pca_df['PC2'],
    pca_df['PC3'],
    s=50,
    alpha=0.7
)

ax.set_title("3D PCA Projection")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.set_zlabel("Principal Component 3")

plt.show()

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
wcss = []

for k in range(1,11):
    kmeans = KMeans(n_clusters=k,random_state=42)
    kmeans.fit_predict(pca_result)
    wcss.append(kmeans.inertia_)
    
    

In [ ]:
wcss

In [ ]:
pip install Kneed

In [ ]:
from kneed import KneeLocator
knee = KneeLocator(range(1,11),wcss,curve="convex",direction="decreasing")

In [ ]:
optimal_k= knee.elbow
optimal_k

In [ ]:
sns.lineplot(x=range(1,11),y=wcss,marker='o')

In [ ]:
## Choosing the k value with sillhouette score
from sklearn.metrics import silhouette_score

In [ ]:
scores=[]
for k in range(2,11):
    kmeans = KMeans(n_clusters=k,random_state=42)
    labels = kmeans.fit_predict(pca_result)
    score = silhouette_score(pca_result,labels)
    scores.append(score)

In [ ]:
sns.lineplot(x=range(2,11),y=scores,marker='o')

## Appling Kmeans Algorithm

In [ ]:
kmeans = KMeans(n_clusters=4,random_state=42)
labels = kmeans.fit_predict(pca_result)

In [ ]:
## Doing pca
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Apply PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(df_scaled)


# Create DataFrame (optional)
import pandas as pd
pca_df = pd.DataFrame(
    pca_result,
    columns=['PC1', 'PC2', 'PC3']
)
# Explained variance
print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)
print("Total Variance Explained:",
      pca.explained_variance_ratio_.sum())

# 3D Plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    pca_df['PC1'],
    pca_df['PC2'],
    pca_df['PC3'],
    s=50,
    alpha=0.7,
    c=labels
)

ax.set_title("3D PCA Projection")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.set_zlabel("Principal Component 3")

plt.show()

## Appling Agglomerative Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering

In [ ]:
agg_clf = AgglomerativeClustering(n_clusters =4,linkage="ward")
label_agg = agg_clf.fit_predict(pca_result)

In [ ]:

# 3D Plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    pca_df['PC1'],
    pca_df['PC2'],
    pca_df['PC3'],
    s=50,
    alpha=0.7,
    c=label_agg
)

ax.set_title("3D PCA Projection")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.set_zlabel("Principal Component 3")

plt.show()